# Azerbaijan Before/After Satellite Image Download

Open this notebook in **Google Colab** and run the cells one by one (Shift+Enter for each cell).
It will ask you to sign in to your Google Earth Engine account — simply authorize it with your Google account.

At the end, the `.png` files will be downloaded directly to Colab. You can download them from the **Files** (folder icon) panel on the left.

In [ ]:
# 1) Install the required libraries
!pip install -q earthengine-api geemap

In [ ]:
# 2) Sign in to Earth Engine
import ee
import geemap

# This line will provide a link. Click it, sign in with your Google account, then copy and paste the authorization code.
ee.Authenticate()

# IMPORTANT: Replace 'your-project-id' below with your own GEE project ID.
# If you have never created a project before, visit https://code.earthengine.google.com,
# create a free project, and enter its name here.
ee.Initialize(project='your-project-id')

In [ ]:
# 3) Locations to visualize (add as many as you want)
# format: 'name': [longitude, latitude]
locations = {
    'Aghdam': [47.146, 39.987],
    # 'Aghali_Zangilan': [LATITUDE_HERE, LONGITUDE_HERE],  # Find the coordinates in Google Maps and add them here
}

buffer_meters = 2000  # Radius (in meters) around the center point to retrieve imagery

before_start, before_end = '2019-05-01', '2019-09-30'
after_start, after_end = '2024-05-01', '2024-09-30'

vis_params = {'min': 0, 'max': 3000, 'gamma': 1.2, 'bands': ['B4', 'B3', 'B2']}

In [ ]:
# 4) Function to create a cloud-free composite
def get_composite(region, start_date, end_date):
    s2 = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(region)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .median()
    )
    return s2.select(['B4', 'B3', 'B2']).clip(region)

In [ ]:
# 5) Download before/after images for each location (saved as .png files in Colab)
for name, (lon, lat) in locations.items():
    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(buffer_meters).bounds()

    before_img = get_composite(region, before_start, before_end)
    after_img = get_composite(region, after_start, after_end)

    before_path = f'{name}_2019_before.png'
    after_path = f'{name}_2024_after.png'

    print(f'{name}: downloading before image (2019)...')
    geemap.get_image_thumbnail(before_img, before_path, vis_params, dimensions=800, region=region)

    print(f'{name}: downloading after image (2024)...')
    geemap.get_image_thumbnail(after_img, after_path, vis_params, dimensions=800, region=region)

    print(f'{name} completed: {before_path}, {after_path}')

print('\nDone. You can download the .png files from the Files (folder icon) panel on the left.')

In [ ]:
# 6) (Optional) Display the images here inside the notebook
from IPython.display import Image, display

for name in locations.keys():
    print(f'--- {name}: BEFORE (2019) ---')
    display(Image(f'{name}_2019_before.png'))
    print(f'--- {name}: AFTER (2024) ---')
    display(Image(f'{name}_2024_after.png'))

## Notes

- If the imagery is too cloudy, adjust the `before_start/before_end` or `after_start/after_end` date range slightly (e.g. `'2019-06-01'` - `'2019-08-31'`).
- Be sure to replace the project name in `ee.Initialize(project='your-project-id')` with your own GEE project ID, otherwise you will get an error.
- To add a new location, add another entry to the `locations` dictionary: `'Name': [longitude, latitude],`
- You can obtain the coordinates by right-clicking the location in Google Maps and copying the displayed latitude/longitude (note: Google Maps shows latitude, longitude, whereas the code expects longitude, latitude).